# MathVision Analytics Notebook



## 1) Business context

The notebook now uses real project-style input files instead of synthetic generation.


## 2) Imports and setup


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

DATA_PATHS = {
    "students": "../data/raw/students.csv",
    "tutors": "../data/raw/tutors.csv",
    "historical_matches": "../data/archive/mathvision_historical_matches_named.csv",
}

TOPICS = ["Algebra", "Fractions", "Geometry", "Calculus", "Statistics"]
CURRICULA = ["Local", "IB", "IGCSE"]


## 3) Load data from CSV files


In [ ]:
def parse_slots(slot_value):
    if pd.isna(slot_value):
        return []
    if isinstance(slot_value, list):
        return slot_value
    return [slot.strip() for slot in str(slot_value).split(";") if slot.strip()]


def load_data_and_prepare(path_students, path_tutors, path_matches):
    students_df = pd.read_csv(path_students)
    tutors_df = pd.read_csv(path_tutors)
    historical_matches_df = pd.read_csv(path_matches)

    tutors_df["available_slots"] = tutors_df["available_slots"].apply(parse_slots)

    # Required numeric fields as numeric
    numeric_cols_students = ["grade_level"]
    students_df[numeric_cols_students] = students_df[numeric_cols_students].apply(pd.to_numeric, errors="coerce")

    numeric_cols_tutors = [
        "years_experience",
        "rating",
        "past_success_rate",
        "preferred_min_grade",
        "preferred_max_grade",
    ]
    tutors_df[numeric_cols_tutors] = tutors_df[numeric_cols_tutors].apply(
        pd.to_numeric, errors="coerce"
    )

    # Keep historical match columns numeric and ensure binary flags are numeric ints
    historical_matches_df[[
        "topic_match",
        "curriculum_match",
        "availability_match",
        "grade_gap",
        "tutor_rating",
        "tutor_experience",
        "past_success_rate",
        "match_score_rule",
        "success_label",
    ]] = historical_matches_df[[
        "topic_match",
        "curriculum_match",
        "availability_match",
        "grade_gap",
        "tutor_rating",
        "tutor_experience",
        "past_success_rate",
        "match_score_rule",
        "success_label",
    ]].apply(pd.to_numeric, errors="coerce")

    # Report any missing values to keep notebook deterministic and debuggable
    print("students_df shape:", students_df.shape)
    print("tutors_df shape:", tutors_df.shape)
    print("historical_matches_df shape:", historical_matches_df.shape)
    print("Missing values")
    print("students_df:", students_df.isna().sum().sum())
    print("tutors_df:", tutors_df.isna().sum().sum())
    print("historical_matches_df:", historical_matches_df.isna().sum().sum())

    return students_df, tutors_df, historical_matches_df


students_df, tutors_df, historical_matches_df = load_data_and_prepare(
    DATA_PATHS["students"],
    DATA_PATHS["tutors"],
    DATA_PATHS["historical_matches"],
)

print(students_df.head())
print(tutors_df.head())
print(historical_matches_df.head())



## 4) Exploratory analysis


In [ ]:

fig, axs = plt.subplots(2, 2, figsize=(12, 8))
axs = axs.ravel()

students_df["curriculum"].value_counts().sort_index().plot(kind="bar", ax=axs[0], title="Students by curriculum")
axs[0].set_ylabel("count")

tutors_df["specialty_topic"].value_counts().sort_index().plot(kind="bar", ax=axs[1], title="Tutors by specialty topic")
axs[1].set_ylabel("count")

axs[2].hist(tutors_df["rating"], bins=10)
axs[2].set_title("Tutor rating histogram")
axs[2].set_xlabel("rating")
axs[2].set_ylabel("count")

historical_matches_df["success_label"].value_counts().sort_index().plot(
    kind="bar", ax=axs[3], title="Success label distribution"
)
axs[3].set_xlabel("success_label")
axs[3].set_ylabel("count")

plt.tight_layout()



## 5) Rule-based scoring helper

The matching logic follows the project spec.


In [ ]:

def compute_rule_features(student_row, tutor_row):
    topic_match = int(tutor_row["specialty_topic"] == student_row["weak_topic"])
    curriculum_match = int(tutor_row["primary_curriculum"] == student_row["curriculum"])
    availability_match = int(student_row["requested_slot"] in tutor_row["available_slots"])

    grade_center = (float(tutor_row["preferred_min_grade"]) + float(tutor_row["preferred_max_grade"])) / 2
    grade_gap = abs(float(student_row["grade_level"]) - grade_center)

    tutor_rating = float(tutor_row["rating"])
    tutor_experience = int(tutor_row["years_experience"])
    past_success_rate = float(tutor_row["past_success_rate"])

    match_score_rule = (
        0.35 * topic_match
        + 0.25 * curriculum_match
        + 0.20 * (tutor_rating / 5)
        + 0.10 * availability_match
        + 0.10 * past_success_rate
    )

    return {
        "topic_match": topic_match,
        "curriculum_match": curriculum_match,
        "availability_match": availability_match,
        "grade_gap": grade_gap,
        "tutor_rating": tutor_rating,
        "tutor_experience": tutor_experience,
        "past_success_rate": past_success_rate,
        "match_score_rule": float(match_score_rule),
    }


sample_student = students_df.iloc[0]
sample_tutor = tutors_df.iloc[0]
sample_features = compute_rule_features(sample_student, sample_tutor)

print("Sample student:", sample_student["student_id"], sample_student["curriculum"], sample_student["weak_topic"], sample_student["requested_slot"])
print("Sample tutor:", sample_tutor["tutor_id"], sample_tutor["primary_curriculum"], sample_tutor["specialty_topic"])
print("Computed features + rule score:")
for k, v in sample_features.items():
    print(f"{k}: {v}")



## 6) Train logistic regression success model


In [ ]:

def train_success_model(train_df):
    feature_cols = [
        "topic_match",
        "curriculum_match",
        "availability_match",
        "grade_gap",
        "tutor_rating",
        "tutor_experience",
        "past_success_rate",
        "match_score_rule",
    ]

    numeric_cols = [
        "grade_gap",
        "tutor_rating",
        "tutor_experience",
        "past_success_rate",
        "match_score_rule",
    ]

    X = train_df[feature_cols].copy()
    y = train_df["success_label"].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
    )

    scaler = StandardScaler()
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    X_train_scaled[numeric_cols] = scaler.fit_transform(X_train_scaled[numeric_cols])
    X_test_scaled[numeric_cols] = scaler.transform(X_test_scaled[numeric_cols])

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print(f"Accuracy:  {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall:    {rec:.3f}")
    print(f"F1-score:  {f1:.3f}")

    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(4, 3))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("Confusion Matrix")
    plt.tight_layout()

    return model, scaler, feature_cols


log_reg_model, scaler, FEATURE_COLUMNS = train_success_model(historical_matches_df)



## 7) Ranking engine

Build ranked recommendations for a student by combining rule score + ML probability.


In [ ]:

def _build_explanation(features_row):
    reasons = []
    if features_row["topic_match"] == 1:
        reasons.append("Strong topic match")
    if features_row["curriculum_match"] == 1:
        reasons.append("Curriculum match")
    if features_row["availability_match"] == 1:
        reasons.append("Available in requested slot")
    if features_row["tutor_rating"] >= 4.6:
        reasons.append("High tutor rating")
    if features_row["past_success_rate"] >= 0.8:
        reasons.append("Good past success rate")
    if not reasons:
        return "No direct rule match flags; model probability still contributes"
    return " | ".join(reasons)


def rank_tutors_for_student(student_row, tutors_df, model, scaler, feature_cols):
    feature_rows = []
    for _, tutor_row in tutors_df.iterrows():
        features = compute_rule_features(student_row, tutor_row)
        feature_rows.append(
            {
                "tutor_id": tutor_row["tutor_id"],
                "specialty_topic": tutor_row["specialty_topic"],
                "primary_curriculum": tutor_row["primary_curriculum"],
                "rating": tutor_row["rating"],
                "years_experience": tutor_row["years_experience"],
                **features,
            }
        )

    ranking_df = pd.DataFrame(feature_rows)
    numeric_cols = ["grade_gap", "tutor_rating", "tutor_experience", "past_success_rate", "match_score_rule"]
    X = ranking_df[feature_cols].copy()
    X_scaled = X.copy()
    X_scaled[numeric_cols] = scaler.transform(X_scaled[numeric_cols])

    ranking_df["predicted_success_probability"] = model.predict_proba(X_scaled)[:, 1]
    ranking_df["final_score"] = (
        0.6 * ranking_df["match_score_rule"] +
        0.4 * ranking_df["predicted_success_probability"]
    )
    ranking_df["rule_score"] = ranking_df["match_score_rule"]
    ranking_df["explanation"] = ranking_df.apply(_build_explanation, axis=1)

    ranking_df = ranking_df.sort_values("final_score", ascending=False).reset_index(drop=True)
    return ranking_df[[
        "tutor_id",
        "specialty_topic",
        "primary_curriculum",
        "rating",
        "years_experience",
        "rule_score",
        "predicted_success_probability",
        "final_score",
        "explanation",
    ]]


demo_student = students_df.sample(1, random_state=42).iloc[0]
demo_ranked = rank_tutors_for_student(
    demo_student,
    tutors_df,
    log_reg_model,
    scaler,
    FEATURE_COLUMNS,
)
print("Demo student:", demo_student["student_id"], demo_student["student_name"], demo_student["curriculum"], demo_student["weak_topic"])
display(demo_ranked.head(5))

top5 = demo_ranked.head(5)
plt.figure(figsize=(8, 4))
plt.barh(top5["tutor_id"], top5["final_score"], color="#2c7fb8")
plt.gca().invert_yaxis()
plt.title("Top 5 tutors for one student")
plt.xlabel("final_score")
plt.tight_layout()



## 8) Demo scenarios

Run the ranking for the three requested student profiles.


In [ ]:

def pick_student(curriculum, weak_topic):
    matches = students_df[(students_df["curriculum"] == curriculum) & (students_df["weak_topic"] == weak_topic)]
    if len(matches) == 0:
        print(f"No student found for {curriculum} + {weak_topic}; using first student as fallback")
        return students_df.iloc[0]
    return matches.iloc[0]

scenarios = [
    ("Local", "Fractions"),
    ("IB", "Calculus"),
    ("IGCSE", "Geometry"),
]

for idx, (curriculum, topic) in enumerate(scenarios, start=1):
    student_row = pick_student(curriculum, topic)
    print(f"Scenario {idx}: {student_row['student_id']} | {curriculum} | {topic}")
    ranked = rank_tutors_for_student(student_row, tutors_df, log_reg_model, scaler, FEATURE_COLUMNS)
    display(ranked.head(5))
    print("-" * 70)



## 9) Business interpretation

This flow is stronger than random tutor assignment because scoring is explicit and consistent,
while predictions learn from historical pair performance.



## 10) Limitations and future improvements

- Inputs now come from provided CSV files, not synthetic generation.
- Historical outcomes depend on source data quality and may be biased.
- `available_slots` parsing assumes semi-colon separated slots.
- Future improvements: add retention/satisfaction features and scheduling constraints.
